# IA para Triagem Negativa (Rule-Out) em Mamografia
**Residente:** Rafael Boava Souza (UNIFESP)

---
**Células do notebook:**
1. Preparação, modelo e validação (métricas e gráficos em português)
2. Simulação de impacto clínico
3. Exportação de figuras para manuscrito (inglês)
4. Execução da exportação

In [ ]:
# ==============================================================================
# CÉLULA 1 — PREPARAÇÃO, MODELO E VALIDAÇÃO
# ==============================================================================

# 1. IMPORTS
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from google.colab import drive

# 2. MONTAGEM DO GOOGLE DRIVE
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 3. CONFIGURAÇÃO DE CAMINHOS
DATA_DIR = '/content/drive/MyDrive/Projeto_Mamografia/Dataset_Treinamento_Cientifico'
WEIGHTS_PATH = '/content/drive/MyDrive/Projeto_Mamografia/melhor_modelo_resnet_V2.pth'

# 4. DISPOSITIVO
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"✅ Executando em: {device}")

# ==============================================================================
# 5. DEFINIÇÃO DA ARQUITETURA DO MODELO (RESNET50)
# CORREÇÃO BUG 1: retorna None se pesos não forem encontrados, evitando
#                  resultados silenciosamente incorretos com pesos aleatórios.
# ==============================================================================
def carregar_modelo_unifesp(path_pesos):
    """Carrega a ResNet50 com os pesos salvos.
    Retorna o modelo em modo eval(), ou None se o arquivo não existir.
    """
    model = models.resnet50(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)  # Classe 0: Nódulo, Classe 1: Normal

    if not os.path.exists(path_pesos):
        print(f"❌ Erro: Arquivo de pesos não encontrado em '{path_pesos}'.")
        print("   Verifique o caminho WEIGHTS_PATH e tente novamente.")
        return None  # <-- retorno explícito de None

    model.load_state_dict(torch.load(path_pesos, map_location=device))
    model = model.to(device)
    model.eval()  # Modo de avaliação (trava os pesos)
    print(f"🚀 Sucesso: Pesos carregados de {path_pesos}")
    return model


# ==============================================================================
# 6. PREPARAÇÃO DOS DADOS (VALIDAÇÃO)
# CORREÇÃO BUG 2: val_loader e class_names são inicializados como None antes
#                  do bloco try/except, evitando NameError se a pasta falhar.
# ==============================================================================
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_loader = None    # <-- inicialização segura
class_names = None   # <-- inicialização segura

try:
    val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'), val_transforms)
    val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False)
    class_names = val_dataset.classes
    print(f"📊 Dataset de Validação pronto: {len(val_dataset)} imagens encontradas.")
except Exception as e:
    print(f"❌ Erro ao acessar a pasta de validação: {e}")


# ==============================================================================
# 7. FUNÇÃO PARA GERAR MÉTRICAS COMPLETAS (MATRIZ DE CONFUSÃO E ROC)
# CORREÇÃO BUG 3 (parcial): adiciona validação do ponto de 98% de sensibilidade
#                             e imprime o TPR real atingido, para que o manuscrito
#                             não reporte 98% quando o modelo não chega lá.
# ==============================================================================
def gerar_metricas_finais(model, dataloader):
    """Gera matriz de confusão e curva ROC para o dataloader fornecido."""
    model.eval()
    y_true, y_pred, all_probs = [], [], []

    print("🔎 Analisando imagens...")
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs  = inputs.to(device)
            outputs = model(inputs)

            # Predições e probabilidades
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

            # Probabilidade da classe Nódulo (índice 0) para a curva ROC
            probs = torch.nn.functional.softmax(outputs, dim=1)
            all_probs.extend(probs[:, 0].cpu().numpy())

    # --- PARTE A: MATRIZ DE CONFUSÃO ---
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matriz de Confusão')
    plt.xlabel('Predição IA')
    plt.ylabel('Verdade Real')

    # --- PARTE B: CURVA ROC ---
    # Invertendo labels para que Nódulo (0) seja o evento positivo (1) no ROC
    y_true_binary = [1 if l == 0 else 0 for l in y_true]
    fpr, tpr, thresholds = roc_curve(y_true_binary, all_probs)
    roc_auc = auc(fpr, tpr)

    # Ponto mais próximo de 98% de sensibilidade
    idx = np.argmin(np.abs(tpr - 0.98))
    safe_threshold = thresholds[idx]
    tpr_real = tpr[idx]  # TPR real atingido

    plt.subplot(1, 2, 2)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
    plt.plot(fpr[idx], tpr[idx], 'ro',
             label=f'Ponto Triagem (Sens={tpr_real:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.title('Curva ROC — Validação de Triagem')
    plt.xlabel('Falso Positivo')
    plt.ylabel('Sensibilidade')
    plt.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

    print("\n" + "="*50)
    print("📋 RESUMO TÉCNICO PARA O TRABALHO:")
    print(f"  Área Sob a Curva (AUC):                {roc_auc:.4f}")
    print(f"  Threshold sugerido (triagem segura):   {safe_threshold:.4f}")
    print(f"  Sensibilidade real no ponto:           {tpr_real*100:.1f}%")
    print(f"  Especificidade no ponto de triagem:    {(1-fpr[idx])*100:.1f}%")
    if abs(tpr_real - 0.98) > 0.01:
        print(f"  ⚠️  AVISO: sensibilidade alvo (98%) NÃO atingida.")
        print(f"            Sensibilidade real = {tpr_real*100:.1f}%.")
    print("="*50)
    print("\nRelatório de Classificação:\n")
    print(classification_report(y_true, y_pred, target_names=class_names))


# ==============================================================================
# 8. EXECUÇÃO FINAL
# CORREÇÃO BUG 2 (cont.): verifica se model e val_loader foram carregados
#                           antes de chamar gerar_metricas_finais.
# ==============================================================================
model_ft = carregar_modelo_unifesp(WEIGHTS_PATH)

if model_ft is None:
    print("⛔ Execução interrompida: modelo não foi carregado.")
elif val_loader is None:
    print("⛔ Execução interrompida: dataset de validação não foi carregado.")
else:
    gerar_metricas_finais(model_ft, val_loader)


In [ ]:
# ==============================================================================
# CÉLULA 2 — SIMULAÇÃO DE IMPACTO CLÍNICO
# CORREÇÃO BUG AVISO: formato da VPN ajustado para 2 casas decimais,
#                      mais legível no contexto clínico.
# ==============================================================================

def simular_impacto_unifesp(n_exames=1000, prevalencia=0.01, sens=0.982, esp=0.676):
    """Simula o fluxo de triagem para um lote de mamografias.

    Parâmetros
    ----------
    n_exames    : total de exames no lote
    prevalencia : proporção de exames com achado (nódulo)
    sens        : sensibilidade do modelo no ponto de triagem
    esp         : especificidade do modelo no ponto de triagem
    """
    # 1. Divisão da população
    n_com_achado = int(n_exames * prevalencia)
    n_normais    = n_exames - n_com_achado

    # 2. Atuação da IA
    verdadeiros_positivos = n_com_achado * sens
    falsos_negativos      = n_com_achado * (1 - sens)
    verdadeiros_negativos = n_normais    * esp
    falsos_positivos      = n_normais    * (1 - esp)

    # 3. Métricas de fluxo
    exames_limpos    = verdadeiros_negativos
    exames_revisados = verdadeiros_positivos + falsos_positivos + falsos_negativos
    reducao_carga    = (exames_limpos / n_exames) * 100

    # VPN: P(doença ausente | teste negativo)
    # CORREÇÃO: formato exibido com 2 casas decimais
    vpn_real = (verdadeiros_negativos / (verdadeiros_negativos + falsos_negativos)) * 100

    print(f"🏥 SIMULAÇÃO: Fluxo de {n_exames} Mamografias (Prevalência {prevalencia*100:.1f}%)")
    print("-" * 55)
    print(f"  ✅ Exames 'LIMPOS' (dispensados de laudo): {int(exames_limpos)}")
    print(f"  ⚠️  Exames para a pilha do Radiologista:   {int(exames_revisados)}")
    print(f"  🚀 Redução de Carga de Trabalho:           {reducao_carga:.1f}%")
    print(f"  🛡️  Segurança Clínica (VPN):               {vpn_real:.2f}%")
    print("-" * 55)

    return reducao_carga, vpn_real


# Executa para 1.000 exames com prevalência de 1% (triagem populaciona)
simular_impacto_unifesp(n_exames=1000, prevalencia=0.01)


In [ ]:
# ==============================================================================
# CÉLULA 3 — EXPORTAÇÃO DE FIGURAS PARA MANUSCRITO (inglês)
# CORREÇÃO BUG 3 (principal): Tabela 1 agora calcula métricas
#                               diretamente de y_true/y_pred, em vez de
#                               usar valores hardcoded.
# CORREÇÃO BUG 1 (chamada): a função agora recebe e usa y_true/y_pred/y_probs
#                             para gerar TODAS as figuras dinamicamente.
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)
from google.colab import files


def export_manuscript_figures(y_true, y_pred, y_probs,
                               class_names=('Mass', 'Normal')):
    """
    Gera e faz download de todas as figuras prontas para manuscrito (inglês).

    Parâmetros
    ----------
    y_true      : lista de rótulos verdadeiros (int)
    y_pred      : lista de predições do modelo (int)
    y_probs     : lista de probabilidades da classe Massa/Nódulo (float)
    class_names : nomes das classes; padrão ('Mass', 'Normal')
    """

    # ------------------------------------------------------------------
    # FIGURA 1 — Matriz de Confusão (threshold 0.5)
    # ------------------------------------------------------------------
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Figure 1: Confusion Matrix (Threshold 0.5)',
              fontsize=12, fontweight='bold')
    plt.xlabel('AI Prediction')
    plt.ylabel('Ground Truth (Radiologist)')
    plt.savefig('figure_1_confusion_matrix.png', bbox_inches='tight', dpi=300)
    plt.show()

    # ------------------------------------------------------------------
    # FIGURA 2 — Curva ROC + ponto de triagem
    # ------------------------------------------------------------------
    y_true_binary = [1 if l == 0 else 0 for l in y_true]
    fpr, tpr, thresholds = roc_curve(y_true_binary, y_probs)
    roc_auc = auc(fpr, tpr)

    idx = np.argmin(np.abs(tpr - 0.98))
    safe_threshold = thresholds[idx]
    tpr_real       = tpr[idx]

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot(fpr[idx], tpr[idx], 'ro',
             label=f'Negative Triage Point (Sens={tpr_real:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.title('Figure 2: ROC Curve for Negative Triage',
              fontsize=12, fontweight='bold')
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Sensitivity)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.2)
    plt.savefig('figure_2_roc_curve.png', bbox_inches='tight', dpi=300)
    plt.show()

    # ------------------------------------------------------------------
    # TABELA 1 — Classification Report
    # CORREÇÃO BUG 3: métricas calculadas dinamicamente (não mais hardcoded)
    # ------------------------------------------------------------------
    classes_idx = sorted(set(y_true))
    report_data = []
    for i, cls in enumerate(classes_idx):
        prec = precision_score(y_true, y_pred, labels=[cls], average='macro', zero_division=0)
        rec  = recall_score   (y_true, y_pred, labels=[cls], average='macro', zero_division=0)
        f1   = f1_score       (y_true, y_pred, labels=[cls], average='macro', zero_division=0)
        sup  = sum(1 for t in y_true if t == cls)
        report_data.append([class_names[i], round(prec, 2),
                             round(rec, 2), round(f1, 2), sup])

    acc = accuracy_score(y_true, y_pred)
    report_data.append(['Accuracy', '', '', round(acc, 2), len(y_true)])

    df_report = pd.DataFrame(
        report_data,
        columns=['Class', 'Precision', 'Recall', 'F1-Score', 'Support']
    )

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.axis('tight'); ax.axis('off')
    table = ax.table(cellText=df_report.values,
                     colLabels=df_report.columns,
                     cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 2.0)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#004a88')
    plt.title("Table 1: Classification Performance Metrics",
              fontsize=12, fontweight='bold', pad=20)
    plt.savefig('table_1_classification_report.png',
                bbox_inches='tight', dpi=300)
    plt.show()

    # ------------------------------------------------------------------
    # TABELA 2 — Simulação de Impacto Clínico
    # ------------------------------------------------------------------
    spec_at_triage = 1 - fpr[idx]
    sim_data = []
    for p in [0.01, 0.05]:
        n   = 1000
        ca  = n * p
        tn  = (n - ca) * spec_at_triage
        fn  = ca * (1 - tpr_real)          # usa TPR real, não 0.982 fixo
        reduction = (tn / n) * 100
        npv = (tn / (tn + fn)) * 100
        setting = 'Screening' if p == 0.01 else 'Diagnostic'
        sim_data.append([f"{p*100:.1f}%", setting,
                         f"{reduction:.1f}%", f"{npv:.2f}%"])

    df_sim = pd.DataFrame(
        sim_data,
        columns=['Prevalence', 'Setting', 'Workload Reduction', 'NPV']
    )

    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.axis('tight'); ax.axis('off')
    table_sim = ax.table(cellText=df_sim.values,
                         colLabels=df_sim.columns,
                         cellLoc='center', loc='center')
    table_sim.auto_set_font_size(False)
    table_sim.set_fontsize(10)
    table_sim.scale(1.2, 2.0)
    for (row, col), cell in table_sim.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#2e7d32')
    plt.title("Table 2: Clinical Workflow Impact Simulation (n=1,000)",
              fontsize=12, fontweight='bold', pad=20)
    plt.savefig('table_2_clinical_simulation.png',
                bbox_inches='tight', dpi=300)
    plt.show()

    # ------------------------------------------------------------------
    # DOWNLOAD DE TODOS OS ARQUIVOS
    # ------------------------------------------------------------------
    to_download = [
        'figure_1_confusion_matrix.png',
        'figure_2_roc_curve.png',
        'table_1_classification_report.png',
        'table_2_clinical_simulation.png',
    ]
    print("\n📥 Downloading all manuscript-ready figures...")
    for f in to_download:
        files.download(f)


In [ ]:
# ==============================================================================
# CÉLULA 4 — EXTRAÇÃO DE DADOS E CHAMADA DA EXPORTAÇÃO
# CORREÇÃO BUG 1 (chamada): verifica se model_ft e val_loader existem antes
#                             de extrair dados e chamar export_manuscript_figures.
# ==============================================================================

def extrair_dados_para_exportar(model, dataloader):
    """Coleta y_true, y_pred e probabilidades da classe Massa do dataloader."""
    model.eval()
    y_true, y_pred, all_probs = [], [], []

    print("🔄 Coletando predições para exportação...")
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs  = inputs.to(device)
            outputs = model(inputs)

            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

            probs = torch.nn.functional.softmax(outputs, dim=1)
            all_probs.extend(probs[:, 0].cpu().numpy())

    return y_true, y_pred, all_probs


# Guarda os nomes das classes conforme definidos no dataset
# (assume que class_names foi definido na Célula 1)
export_class_names = class_names if class_names else ['Mass', 'Normal']

# Executa somente se modelo e dataloader estiverem disponíveis
if 'model_ft' not in dir() or model_ft is None:
    print("⛔ Execute a Célula 1 primeiro para carregar o modelo.")
elif val_loader is None:
    print("⛔ Execute a Célula 1 primeiro para carregar o dataset.")
else:
    y_true_vals, y_pred_vals, prob_vals = extrair_dados_para_exportar(
        model_ft, val_loader
    )
    export_manuscript_figures(
        y_true_vals, y_pred_vals, prob_vals,
        class_names=export_class_names
    )
